# FIT3182 Assignment 2 — Task 1: MongoDB Data Model

**Student IDs:** 34080678 | 33590982

This notebook implements **Task 1** of Assignment 2, covering the design and creation of MongoDB collections for the traffic speed camera system.

- **Task 1.1** — Collection Design: Vehicle, Camera, and Violation collections (schema, sample document, indexes, shard key, retention policy)
- **Task 1.2** — Collection Relationships: embed vs. reference justification, read/write patterns, trade-offs

**Dataset overview:**

| File | Description |
|------|-------------|
| `vehicle.csv` | Registered vehicles (plate, owner, type) |
| `camera.csv` | Speed camera metadata (location, speed limit) |
| `camera_event_A/B/C.csv` | Real-time speed reading events from 3 producers (used in Task 2+) |
| `camera_event_historic.csv` | Historic speed violations (used in Task 2+) |


In [ ]:
# ── Installation & Imports ──────────────────────────────────────────────────
# pymongo is pre-installed in this environment.
# To reinstall: python -m pip install pymongo --target <venv>/Lib/site-packages

import pymongo
from pymongo import MongoClient, ASCENDING
import pandas as pd
import json
import pathlib

# change the data directory path if your setup differs
DATA_DIR = pathlib.Path("/home/student/Assignment 2/FIT3182/data")

print(f"pymongo version: {pymongo.__version__}")
print("All required libraries imported successfully.")
print(f"DATA_DIR: {DATA_DIR}  (exists={DATA_DIR.exists()})")

pymongo version: 4.3.3
All required libraries imported successfully.
DATA_DIR: /home/student/Assignment 2/FIT3182/data  (exists=True)


## MongoDB Connection

Connect to the MongoDB instance running in Docker on port **27017**.  
The database `fit3182_db` is used throughout this assignment.

In [ ]:
# ── MongoDB Connection ───────────────────────────────────────────────────────
# Both containers are on Docker's default bridge network; name-based DNS is
# not supported there, so we use fervent_hamilton's bridge IP directly.
# get your MongoDB IP using docker inspect fervent_hamilton --format "{{.NetworkSettings.IPAddress}}" 
MONGO_HOST = "172.17.0.2"
MONGO_PORT = 27017
DB_NAME    = "fit3182_db"

client = MongoClient(MONGO_HOST, MONGO_PORT, serverSelectionTimeoutMS=5000)

# Verify connectivity
try:
    client.admin.command("ping")
    print(f"Connected to MongoDB at {MONGO_HOST}:{MONGO_PORT}")
except Exception as e:
    print(f"Connection failed: {e}")
    raise

db = client[DB_NAME]
print(f"Using database: '{DB_NAME}'")

Connected to MongoDB at 172.17.0.2:27017
Using database: 'fit3182_db'


---
## Task 1.1 — Collection Design

Three collections are designed for this system:

| Collection | Purpose |
|---|---|
| `vehicle` | Registered vehicle and owner information |
| `camera` | Speed camera locations and speed limits |
| `violation` | Detected speeding violations (written by streaming pipeline in Task 2) |

---
### Collection 1: `vehicle`

**Description:**  
Stores registered vehicle data including the owner's details and vehicle type. The `car_plate` field uniquely identifies each vehicle and acts as a natural primary key and the foreign reference used in the `violation` collection.

**Document Schema:**

```json
{
  "car_plate":         "string  — unique licence plate (PK)",
  "owner_name":        "string  — full name of registered owner",
  "owner_addr":        "string  — residential address",
  "vehicle_type":      "string  — e.g. Sedan, SUV, Coupe, Hatchback",
  "registration_date": "Date    — ISO 8601 date of registration"
}
```

**Sample Document:**
```json
{
  "car_plate": "ABC123",
  "owner_name": "Jane Smith",
  "owner_addr": "12 Example St, Melbourne VIC 3000",
  "vehicle_type": "Sedan",
  "registration_date": "2024-01-01T00:00:00Z"
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `car_plate` | Unique ascending | Primary lookup key; enforces uniqueness |
| `vehicle_type` | Ascending | Supports analytics queries filtered by vehicle type |

**Shard Key:** `car_plate` — high cardinality, evenly distributed, used in cross-collection joins.

**Retention Policy:** Records are kept indefinitely; vehicles deregistered > 7 years ago may be archived to cold storage.


In [13]:
# ── Collection 1: vehicle ────────────────────────────────────────────────────
# Drop and recreate for a clean run
db.drop_collection("vehicle")

# Create collection with JSON Schema validation
db.create_collection("vehicle", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["car_plate", "owner_name", "vehicle_type", "registration_date"],
        "properties": {
            "car_plate":         {"bsonType": "string",   "description": "Unique licence plate"},
            "owner_name":        {"bsonType": "string",   "description": "Registered owner name"},
            "owner_addr":        {"bsonType": "string",   "description": "Owner address"},
            "vehicle_type":      {"bsonType": "string",   "enum": ["Sedan","SUV","Coupe","Hatchback","Van","Truck"]},
            "registration_date": {"bsonType": "date",     "description": "ISO date of registration"},
        }
    }
})

# Create indexes
vehicle_col = db["vehicle"]
vehicle_col.create_index([("car_plate", ASCENDING)], unique=True, name="idx_car_plate_unique")
vehicle_col.create_index([("vehicle_type", ASCENDING)],            name="idx_vehicle_type")
print("Indexes created:", list(vehicle_col.index_information().keys()))

# Load data from CSV
# Note: source CSV has a typo 'vechicle_type' — corrected to 'vehicle_type' on load
df_vehicle = pd.read_csv(DATA_DIR / "vehicle.csv")
df_vehicle.rename(columns={"vechicle_type": "vehicle_type"}, inplace=True)
df_vehicle["registration_date"] = pd.to_datetime(df_vehicle["registration_date"])
# Deduplicate on car_plate, keeping first occurrence
df_vehicle.drop_duplicates(subset="car_plate", keep="first", inplace=True)

records = df_vehicle.to_dict(orient="records")
result = vehicle_col.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} vehicle documents (after deduplication)")

# Display a sample document
sample = vehicle_col.find_one({}, {"_id": 0})
print("\nSample document:")
print(json.dumps({k: str(v) for k, v in sample.items()}, indent=2))

Indexes created: ['_id_', 'idx_car_plate_unique', 'idx_vehicle_type']
Inserted 9844 vehicle documents (after deduplication)

Sample document:
{
  "car_plate": "FT 02",
  "owner_name": "Goh Mei Wei",
  "owner_addr": "943 Jalan Bukit Mawar, Kuala Lumpur",
  "vehicle_type": "Coupe",
  "registration_date": "2006-08-22 03:18:00"
}


---
### Collection 2: `camera`

**Description:**  
Stores speed camera metadata including GPS coordinates, road position, and the enforced speed limit. This is a small, **read-heavy, rarely updated** reference collection. It is referenced by both `camera_event` and `violation` to avoid duplicating speed-limit data.

**Document Schema:**

```json
{
  "camera_id":    "int     — unique camera identifier (PK)",
  "latitude":     "double  — GPS latitude",
  "longitude":    "double  — GPS longitude",
  "position":     "double  — road position marker (km)",
  "speed_limit":  "int     — enforced speed limit (km/h)"
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `camera_id` | Unique ascending | Primary lookup; joins from events and violations |
| `speed_limit` | Ascending | Supports queries filtering cameras by speed zone |

**Shard Key:** `camera_id` — monotonically increasing integer; suitable for range-based sharding.

**Retention Policy:** Permanent — camera records represent physical infrastructure. Decommissioned cameras should be marked inactive rather than deleted (to preserve historical violation context).

In [14]:
# ── Collection 2: camera ─────────────────────────────────────────────────────
db.drop_collection("camera")

db.create_collection("camera", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["camera_id", "latitude", "longitude", "speed_limit"],
        "properties": {
            "camera_id":   {"bsonType": "int",    "description": "Unique camera ID"},
            "latitude":    {"bsonType": "double", "description": "GPS latitude"},
            "longitude":   {"bsonType": "double", "description": "GPS longitude"},
            "position":    {"bsonType": "double", "description": "Road position marker (km)"},
            "speed_limit": {"bsonType": "int",    "description": "Enforced speed limit (km/h)"},
        }
    }
})

camera_col = db["camera"]
camera_col.create_index([("camera_id", ASCENDING)], unique=True, name="idx_camera_id_unique")
camera_col.create_index([("speed_limit", ASCENDING)],             name="idx_speed_limit")
print("Indexes created:", list(camera_col.index_information().keys()))

# Load data from CSV
df_camera = pd.read_csv(DATA_DIR / "camera.csv")
df_camera["camera_id"]   = df_camera["camera_id"].astype(int)
df_camera["speed_limit"] = df_camera["speed_limit"].astype(int)

records = df_camera.to_dict(orient="records")
result = camera_col.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} camera documents")

sample = camera_col.find_one({}, {"_id": 0})
print("\nSample document:")
print(json.dumps(sample, indent=2))

Indexes created: ['_id_', 'idx_camera_id_unique', 'idx_speed_limit']
Inserted 3 camera documents

Sample document:
{
  "camera_id": 1,
  "latitude": 2.157730731,
  "longitude": 102.6601002,
  "position": 152.5,
  "speed_limit": 110
}


---
### Collection 3: `violation`

**Description:**  
Stores confirmed speeding violations detected by the real-time streaming pipeline (Task 2+). Each document represents one vehicle on one date and contains an embedded array of individual violation events for that day. This design minimises document count while keeping all daily violations co-located for fast per-plate per-day queries.

This collection is **intentionally left empty** at Task 1 setup — violations will be written here by the Spark Structured Streaming consumer in the next task.

**Document Schema:**

```json
{
  "car_plate": "string  — reference to vehicle.car_plate",
  "date":      "Date    — calendar date of violations (midnight UTC)",
  "violations": [
    {
      "violation_type":  "string  — 'INSTANTANEOUS' or 'AVERAGE'",
      "camera_id_start": "int     — entry camera (same as end for instantaneous)",
      "camera_id_end":   "int     — exit camera",
      "timestamp_start": "Date    — event time at entry camera",
      "timestamp_end":   "Date    — event time at exit camera",
      "speed_reading":   "double  — recorded or computed speed (km/h)"
    }
  ]
}
```

**Sample Document:**
```json
{
  "car_plate": "ABC123",
  "date": "2024-01-01T00:00:00Z",
  "violations": [
    {
      "violation_type": "INSTANTANEOUS",
      "camera_id_start": 1,
      "camera_id_end": 1,
      "timestamp_start": "2024-01-01T08:00:04Z",
      "timestamp_end": "2024-01-01T08:00:04Z",
      "speed_reading": 125.0
    },
    {
      "violation_type": "AVERAGE",
      "camera_id_start": 1,
      "camera_id_end": 2,
      "timestamp_start": "2024-01-01T08:00:00Z",
      "timestamp_end": "2024-01-01T08:00:33Z",
      "speed_reading": 118.2
    }
  ]
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `car_plate, date` | Compound ascending (unique) | Primary upsert key; one document per vehicle per day |
| `date` | Ascending | Time-range queries across all vehicles |

**Shard Key:** `car_plate` — consistent with `vehicle` collection; co-locates a vehicle's documents across shards.

**Retention Policy:** Violations are legally significant records. Retain indefinitely. A TTL index on an `archived_at` field (e.g. 7 years) could be applied for regulatory purging if required.


In [15]:
# ── Collection 3: violation ──────────────────────────────────────────────────
db.drop_collection("violation")

db.create_collection("violation", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["car_plate", "date", "violations"],
        "properties": {
            "car_plate":  {"bsonType": "string", "description": "Ref: vehicle.car_plate"},
            "date":       {"bsonType": "date",   "description": "Calendar date of violations (midnight UTC)"},
            "violations": {
                "bsonType": "array",
                "description": "Embedded array of violation events for this vehicle/day",
                "items": {
                    "bsonType": "object",
                    "required": ["violation_type", "camera_id_start", "camera_id_end",
                                 "timestamp_start", "timestamp_end", "speed_reading"],
                    "properties": {
                        "violation_type":  {"bsonType": "string", "enum": ["INSTANTANEOUS", "AVERAGE"]},
                        "camera_id_start": {"bsonType": "int"},
                        "camera_id_end":   {"bsonType": "int"},
                        "timestamp_start": {"bsonType": "date"},
                        "timestamp_end":   {"bsonType": "date"},
                        "speed_reading":   {"bsonType": "double"},
                    }
                }
            }
        }
    }
})

violation_col = db["violation"]
# Compound unique index — one document per vehicle per day; supports upsert in Task 2
violation_col.create_index(
    [("car_plate", ASCENDING), ("date", ASCENDING)],
    unique=True, name="idx_plate_date_unique"
)
violation_col.create_index([("date", ASCENDING)], name="idx_date")
print("Indexes created:", list(violation_col.index_information().keys()))

# Collection is intentionally empty at this stage.
# Violations will be written by the Spark Structured Streaming pipeline in Task 2.
print(f"\nviolation collection created with {violation_col.count_documents({})} documents (empty — ready for Task 2)")


Indexes created: ['_id_', 'idx_plate_date_unique', 'idx_date']

violation collection created with 0 documents (empty — ready for Task 2)


---
## Task 1.2 — Collection Relationships

### Relationship Diagram

```
vehicle (car_plate PK)
    │
    └──── referenced by ──── violation.car_plate

camera (camera_id PK)
    │
    ├──── referenced by ──── violation.violations[].camera_id_start
    └──── referenced by ──── violation.violations[].camera_id_end
```

---

### Design Decision: Reference vs Embed

---

#### `violation` → `vehicle` (via `car_plate`)

**Approach: Reference**

- Violations are **written once and read many times** (reports, dashboards, enforcement).
- Owner details can change (address update, ownership transfer). Referencing ensures queries always return current owner information without backfilling violation documents.
- The trade-off is a two-collection lookup (`violation` → `vehicle`), but this is acceptable since violation queries are typically filtered first by `car_plate` or `date`, and the vehicle lookup is a single-key point query on an indexed field.

---

#### `violation` → `camera` (via `camera_id_start` / `camera_id_end`)

**Approach: Reference (with daily violations embedded inside `violation`)**

- There are only 3 cameras — the `camera` collection is tiny and fits entirely in memory.
- Referencing avoids duplicating speed-limit values across every violation sub-document.
- Camera metadata (location, speed limit) is stable reference data; a join at query time is negligible.

---

#### Violations embedded within a daily `violation` document

**Approach: Embed**

- Each `violation` document groups all events for a given `car_plate` + `date` pair. Individual violation events are **embedded** as an array rather than stored as separate top-level documents.
- This pattern minimises document count and aligns with the streaming upsert pattern: the Spark consumer will `$push` new events into the existing array using a single `update_one(..., upsert=True)` call per batch, avoiding expensive per-event document creation.
- Trade-off: if a vehicle accumulates many violations in a single day the array could grow large, but in practice daily violations per vehicle are rare enough that this is not a concern.

---

### Consistency Considerations

MongoDB does not enforce foreign-key constraints. Application-level consistency is maintained by:

1. **Insert order:** `vehicle` and `camera` documents are loaded before any `violation` documents are written by the streaming pipeline.
2. **Validation schemas** on each collection reject documents with missing required fields.
3. **Unique compound index** on `(car_plate, date)` prevents duplicate daily documents from stream retries (idempotent upserts).

---

### Summary Table

| Relationship | Strategy | Reason |
|---|---|---|
| `violation.car_plate` → `vehicle` | Reference | Owner data changes; reference ensures freshness |
| `violation.violations[].camera_id_*` → `camera` | Reference | Camera list is tiny static reference data |
| Individual violation events inside `violation` | Embed | Groups daily events for efficient upsert; low growth risk |


In [16]:
# ── Database Summary Verification ────────────────────────────────────────────
print(f"Collections in '{DB_NAME}':")
print("-" * 45)
for col_name in sorted(db.list_collection_names()):
    col = db[col_name]
    count = col.count_documents({})
    indexes = list(col.index_information().keys())
    print(f"  {col_name:<18} {count:>6} documents   indexes: {indexes}")

print("\nTask 1 setup complete.")

Collections in 'fit3182_db':
---------------------------------------------
  FIT                     3 documents   indexes: ['_id_']
  camera                  3 documents   indexes: ['_id_', 'idx_camera_id_unique', 'idx_speed_limit']
  vehicle              9844 documents   indexes: ['_id_', 'idx_car_plate_unique', 'idx_vehicle_type']
  violation               0 documents   indexes: ['_id_', 'idx_plate_date_unique', 'idx_date']

Task 1 setup complete.
